<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Exercises_XP_RAG_W7D3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: RAG with LangChain (Student)

## 0) Setup


In [ ]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

In [ ]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA


## 1) Load dataset and convert to Documents


In [ ]:
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"
text_column = "text"
source_column = "source"

# Chargement du jeu de données depuis Hugging Face
ds = load_dataset(dataset_name, split=split)

documents: List[Document] = []
# On parcourt chaque ligne du dataset pour créer des objets 'Document' utilisables par LangChain
for i, row in enumerate(ds):
    documents.append(
        Document(
            page_content=row[text_column], # Le contenu textuel principal de la page
            metadata={"source": row[source_column]} # Les métadonnées pour garder une trace de l'origine
        )
    )

print("Documents:", len(documents))
print("Example:", documents[0].metadata)
print(documents[0].page_content[:350])

## 2) Split into chunks


In [ ]:
# On définit la taille des morceaux (chunks). 500 caractères est un bon compromis.
chunk_size = 500
# On garde un chevauchement (overlap) pour ne pas couper les phrases en plein milieu du sens.
chunk_overlap = 50

# L'outil qui va découper intelligemment le texte en fonction des paragraphes et caractères
splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

# On applique le découpage à notre liste de documents
splits = splitter.split_documents(documents)
print("Chunks:", len(splits))
print("First chunk:", splits[0].metadata)
print(splits[0].page_content[:350])

## 3) Vector store + retriever (FAISS)


In [ ]:
from langchain_community.vectorstores import FAISS, DistanceStrategy

# Modèle pour transformer du texte en vecteurs mathématiques
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

# Création de la base de données vectorielle avec FAISS
vectorstore = FAISS.from_documents(
    documents=splits, # On donne nos morceaux de texte découpés
    embedding=embeddings, # On donne le modèle pour les traduire en vecteurs
    distance_strategy=DistanceStrategy.COSINE # On utilise la similarité cosinus pour comparer les sens
)

# Le 'retriever' est l'outil qui ira chercher les 4 morceaux les plus proches d'une question
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready")

Retriever ready


## 4) Build the RAG chain


In [ ]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA

llm_id = "google/flan-t5-small"
# Configuration du pipeline Hugging Face pour la génération de texte
hf = pipeline(
    task="text2text-generation", # Type de tâche pour Flan-T5
    model=llm_id,
    max_new_tokens=100 # On limite la longueur de la réponse
)

# On encapsule le pipeline dans un objet LangChain
llm = HuggingFacePipeline(pipeline=hf)

# Création de la chaîne de QA (Question/Réponse)
qa = RetrievalQA.from_chain_type(
    llm=llm, # Le cerveau (le modèle de langue)
    retriever=retriever, # La bibliothèque (notre base de données)
    chain_type="stuff", # On 'fourre' (stuff) simplement les docs trouvés dans le prompt
    return_source_documents=True # Pour afficher d'où viennent les informations
)

print("RAG chain ready")

## 5) Demo: RAG vs no-RAG


In [ ]:
q = "How can I retrieve a model from the Hugging Face Hub?"

# --- Test sans RAG (Le modèle répond avec ses propres connaissances limitées) ---
no_rag_prompt = (
    "Answer the question. If you are not sure, say you are not sure.\n\n"
    f"Question: {q}\n"
    "Answer:"
)
# Appel direct au modèle de base
no_rag_answer = hf(no_rag_prompt)[0]["generated_text"]

# --- Test avec RAG (Le modèle utilise nos documents comme source) ---
rag_result = qa({"query": q}) # On envoie la question à la chaîne QA

print("Q:", q)
print("\nNo-RAG answer:\n", no_rag_answer)
print("\nRAG answer:\n", rag_result["result"])
print("\nSources:")
# On affiche les métadonnées 'source' pour prouver d'où vient l'info
for d in rag_result["source_documents"]:
    print("-", d.metadata.get("source"))